# Resume / Candidate Screening System (Task 3)
### End-to-End Walkthrough with Kaggle Resume Dataset

**Objective**:
Demonstrate an explainable resume screening pipeline that:
1. Loads resumes directly from the local **Kaggle Resume Dataset** (`Resume.csv`).
2. Accepts a user-provided job description.
3. Cleans text and heuristically redacts contact info (emails, phone numbers, URLs).
4. Extracts technical skills from both the job description and candidate resumes.
5. Computes semantic text similarity (TF-IDF unigram+bigram cosine similarity) and skill overlap metrics.
6. Generates explainable match scores, rankings, and natural-language explanations.

> **Responsible AI Notice**: Scores reflect textual and skill alignment with the user-provided job description, not human character or hiring suitability. Category labels are strictly metadata and never used as ranking inputs.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd

from src.dataset import resolve_kaggle_dataset_dir, load_kaggle_resumes, get_kaggle_pdf_dir
from src.preprocessing import clean_text, extract_text_from_file
from src.skill_extraction import extract_skills, extract_categorized_skills
from src.matching import calculate_match
from src.ranking import rank_candidates, candidates_to_dataframe

print("All modules loaded successfully.")
print("Project root:", repo_root)

## Step 1: Discover and Load Resumes from Kaggle Dataset
Loads records from `%USERPROFILE%\Downloads\resume dataset\Resume\Resume.csv`.

In [ ]:
kaggle_dir = resolve_kaggle_dataset_dir()
if not kaggle_dir:
    raise FileNotFoundError(
        "Kaggle Resume Dataset not found. Please extract the dataset to: "
        "%USERPROFILE%\\Downloads\\resume dataset\\Resume\\Resume.csv"
    )

print(f"Discovered local Kaggle Resume Dataset at: {kaggle_dir}")

# Load resumes from INFORMATION-TECHNOLOGY and ENGINEERING for screening
resumes_df, status = load_kaggle_resumes(
    limit=10,
    categories=["INFORMATION-TECHNOLOGY", "ENGINEERING"],
)
print(status)
print(f"Loaded {len(resumes_df)} resumes. Columns: {resumes_df.columns.tolist()}")
resumes_df[["ID", "Category"]].head()

## Step 2: Accept User-Provided Job Description
The user provides the target role requirements. We extract the required skills.

In [ ]:
user_job_description = """
Job Title: Machine Learning & Python Engineer

Requirements:
- 3+ years experience in Python and SQL.
- Strong background in Machine Learning, Deep Learning, and PyTorch or TensorFlow.
- Practical knowledge of Scikit-Learn for model prototyping and feature engineering.
- Containerization experience using Docker and orchestration with Kubernetes.
- Cloud deployment experience on AWS.
- Familiarity with Git, Linux, and CI/CD pipelines.
- Excellent problem solving and communication skills.
"""

cleaned_jd = clean_text(user_job_description, redact=False)
required_skills = extract_skills(cleaned_jd)

print(f"Required Skills extracted from Job Description ({len(required_skills)} total):")
print(required_skills)

## Step 3: Text Preprocessing & Contact Information Redaction
Clean resume text and apply heuristic redaction for emails, phone numbers, and URLs.

In [ ]:
cleaned_candidates = {}
for _, row in resumes_df.iterrows():
    cand_id = f"Candidate_{row['ID']} ({row['Category']})"
    cleaned_candidates[cand_id] = clean_text(row["Resume_str"], redact=True)

first_key = list(cleaned_candidates.keys())[0]
print(f"=== Cleaned Preview for {first_key} ===")
print(cleaned_candidates[first_key][:350] + "...")

## Step 4: Compare Resumes and Compute Explainable Match Scores
Calculates required-skill coverage, Jaccard similarity, and TF-IDF cosine similarity.

In [ ]:
match_results = []
for cand_id, resume_text in cleaned_candidates.items():
    res = calculate_match(
        resume_text=resume_text,
        job_description=cleaned_jd,
        candidate_id=cand_id,
        skill_weight=0.60,
        text_weight=0.40,
    )
    match_results.append(res)

print(f"Evaluated {len(match_results)} candidate resumes.")

## Step 5: Rank Candidates & Inspect Explanations
Candidates are ranked from highest to lowest score with deterministic tie-breaking.

In [ ]:
ranked_candidates = rank_candidates(match_results)
ranking_df = candidates_to_dataframe(ranked_candidates)

display_cols = [
    "Rank", "Candidate ID", "Overall Match Score (%)",
    "Skill Coverage (%)", "Text Similarity (0-1)",
    "Matched Skills Count", "Missing Skills Count"
]

print("=== Screening Results: Leaderboard ===")
print(ranking_df[display_cols].to_string(index=False))

print("\n=== Top Ranked Candidate Explanation ===")
top_cand = ranked_candidates[0]
print(top_cand.explanation)
print("Matched Skills:", top_cand.matched_skills)
print("Missing Skills:", top_cand.missing_skills)

## Step 6: In-Memory PDF Extraction Demonstration
Demonstrating in-memory PDF parsing using PyMuPDF without writing files to disk.

In [ ]:
pdf_dir = get_kaggle_pdf_dir()
if pdf_dir and pdf_dir.exists():
    it_pdf_dir = pdf_dir / "INFORMATION-TECHNOLOGY"
    sample_pdf = next(it_pdf_dir.glob("*.pdf"), None)
    if sample_pdf:
        print(f"Found sample PDF: {sample_pdf.name} ({sample_pdf.stat().st_size} bytes)")
        pdf_bytes = sample_pdf.read_bytes()
        result = extract_text_from_file(pdf_bytes, filename=sample_pdf.name)
        print(f"In-Memory Extraction Success: {result.success}")
        print(f"Cleaned Text Length: {len(result.text)} characters")
        
        # Screen this PDF against the user job description
        pdf_match = calculate_match(
            resume_text=result.text,
            job_description=cleaned_jd,
            candidate_id=f"PDF_{sample_pdf.stem}",
        )
        print(f"Match Score: {pdf_match.overall_score:.1f}%")
        print(f"Matched Skills: {pdf_match.matched_skills}")
        print(f"Missing Skills: {pdf_match.missing_skills}")

## Summary
- Processed resumes directly from the local Kaggle dataset.
- Extracted skills deterministically using the curated ontology.
- Generated explainable match scores decomposed into skill coverage, missing skills, and text similarity.
- Maintained candidate privacy with in-memory parsing and zero disk persistence.